# 02 — Cleaning

Build the reproducible Python ETL pipeline for the US Accidents dataset. Every transformation is logged, and the final cleaned dataset is exported to `data/processed/cleaned_dataset.csv` for downstream EDA and statistical analysis.

**Cleaning is organised around the five problem-statement dimensions:**

1. **Severity** — target variable, must be non-null and typed correctly.
2. **Geography** — state, city, lat/lng preserved for hotspot analysis.
3. **Time** — derive hour, day-of-week, month, season, rush-hour, weekend flags.
4. **Weather** — numeric weather variables imputed with medians (robust to skew).
5. **Infrastructure** — POI booleans (traffic signal, junction, crossing, …) forced to bool.

All step-level logic lives in `scripts/etl_pipeline.py` so it can also be run from the CLI.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.etl_pipeline import (
    load_us_accidents,
    drop_unused_columns,
    parse_datetimes,
    drop_missing_critical,
    impute_weather,
    impute_categorical,
    coerce_booleans,
    add_duration,
    add_time_features,
    add_severity_label,
    handle_outliers,
    save_processed,
    DEFAULT_ANALYSIS_SAMPLE_N,
    DROP_COLUMNS,
    NUMERIC_WEATHER_COLUMNS,
    CATEGORICAL_FILL_COLUMNS,
    BOOLEAN_POI_COLUMNS,
)

RAW_PATH       = PROJECT_ROOT / 'data' / 'raw' / 'US_Accidents_March23.csv'
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_dataset.csv'

# Tableau Public works best with a smaller extract. We keep the full raw file
# locally, but clean and analyze a reproducible representative sample.
SAMPLE_N: int | None = DEFAULT_ANALYSIS_SAMPLE_N

## Step 1 — Load a representative raw sample & normalize column names

The full combined CSV remains in `data/raw/` for reproducibility. This notebook cleans a fixed 100,000-row random sample to produce a small, easy-to-share dataset for the analysis teammate.

In [2]:
df = load_us_accidents(RAW_PATH, sample_n=SAMPLE_N)
rows_initial, cols_initial = df.shape
print(f'Loaded: {rows_initial:,} rows × {cols_initial} columns')
df.head(3)

Loaded: 100,000 rows × 46 columns


,id,source,severity,start_time,end_time,start_lat,start_lng,end_lat,end_lng,distance_mi,...,roundabout,station,stop,traffic_calming,traffic_signal,turning_loop,sunrise_sunset,civil_twilight,nautical_twilight,astronomical_twilight
0,A-804827,Source2,3,2021-11-24 19:36:02,2021-11-24 20:35:31,32.869389,-96.670082,NaN,NaN,0.000,...,False,False,False,False,False,False,Night,Night,Night,Night
1,A-3362792,Source2,2,2017-08-08 08:52:45,2017-08-08 09:38:00,40.099434,-75.196213,NaN,NaN,0.000,...,False,False,False,False,True,False,Day,Day,Day,Day
2,A-3533324,Source1,2,2016-10-24 16:09:01,2016-10-24 22:09:01,34.143244,-117.256491,34.14364,-117.259669,0.184,...,False,False,False,False,False,False,Day,Day,Day,Day


## Step 2 — Drop columns not used in the government severity analysis

| Column | Reason dropped |
|---|---|
| `id`, `source` | internal identifiers |
| `description` | free text, not structured analysis |
| `end_lat`, `end_lng` | ~50% null, redundant with start coordinates |
| `number` | building number, ~60% null, not analytical |
| `airport_code`, `weather_timestamp` | audit metadata for the weather join |
| `country` | constant = `US` |
| `turning_loop` | constant = `False` in this dataset |
| `nautical_twilight`, `astronomical_twilight` | redundant with `sunrise_sunset` / `civil_twilight` |

In [3]:
df = drop_unused_columns(df)
print(f'After drop: {df.shape[1]} columns')
print('Columns remaining:', list(df.columns))

After drop: 35 columns
Columns remaining: ['severity', 'start_time', 'end_time', 'start_lat', 'start_lng', 'distance_mi', 'street', 'city', 'county', 'state', 'zipcode', 'timezone', 'temperature_f', 'wind_chill_f', 'humidity', 'pressure_in', 'visibility_mi', 'wind_direction', 'wind_speed_mph', 'precipitation_in', 'weather_condition', 'amenity', 'bump', 'crossing', 'give_way', 'junction', 'no_exit', 'railway', 'roundabout', 'station', 'stop', 'traffic_calming', 'traffic_signal', 'sunrise_sunset', 'civil_twilight']


## Step 3 — Parse datetimes and drop rows missing critical fields

`severity`, `start_time`, `state`, `start_lat`, `start_lng` are required for every downstream KPI. Rows missing any of these are dropped.

In [4]:
df = parse_datetimes(df)
before = len(df)
df = drop_missing_critical(df)
print(f'Dropped {before - len(df):,} rows missing critical fields ({(before - len(df)) / before * 100:.2f}%)')

Dropped 9,674 rows missing critical fields (9.67%)


## Step 4 — Impute missing weather values with column medians

Weather variables are right-skewed, so medians are more robust than means.

In [5]:
missing_before = df[list(NUMERIC_WEATHER_COLUMNS)].isna().sum()
df = impute_weather(df)
missing_after = df[list(NUMERIC_WEATHER_COLUMNS)].isna().sum()
pd.DataFrame({'missing_before': missing_before, 'missing_after': missing_after})

,missing_before,missing_after
temperature_f,1881,0
wind_chill_f,25648,0
humidity,2000,0
pressure_in,1662,0
visibility_mi,2089,0
wind_speed_mph,7131,0
precipitation_in,28161,0


## Step 5 — Fill categorical nulls with `Unknown`

Preserves rows for geographic and temporal aggregations.

In [6]:
df = impute_categorical(df)
cat_nulls = df[[c for c in CATEGORICAL_FILL_COLUMNS if c in df.columns]].isna().sum()
print('Remaining categorical nulls:')
print(cat_nulls)

Remaining categorical nulls:
wind_direction       0
weather_condition    0
zipcode              0
city                 0
county               0
street               0
timezone             0
sunrise_sunset       0
civil_twilight       0
dtype: int64


## Step 6 — Coerce infrastructure POI flags to boolean

In [7]:
df = coerce_booleans(df)
print('Boolean POI dtypes:')
print(df[[c for c in BOOLEAN_POI_COLUMNS if c in df.columns]].dtypes)

Boolean POI dtypes:
amenity            bool
bump               bool
crossing           bool
give_way           bool
junction           bool
no_exit            bool
railway            bool
roundabout         bool
station            bool
stop               bool
traffic_calming    bool
traffic_signal     bool
dtype: object


## Step 7 — Compute `duration_min` and drop impossible durations

Negative durations (end before start) or > 24 h are data-entry errors and are removed.

In [8]:
before = len(df)
df = add_duration(df)
print(f'Dropped {before - len(df):,} rows with invalid duration')
print('Duration distribution (min):')
print(df['duration_min'].describe().round(2))

Dropped 406 rows with invalid duration
Duration distribution (min):
count    89920.00
mean       106.66
std        124.99
min          5.50
25%         30.00
50%         62.50
75%        122.30
max       1440.00
Name: duration_min, dtype: float64


## Step 8 — Derive time-based features

`hour`, `day_of_week`, `day_name`, `month`, `month_name`, `year`, `season`, `is_weekend`, `is_rush_hour` — these feed the EDA charts and the Tableau dashboard filters.

In [9]:
df = add_time_features(df)
df[['start_time', 'hour', 'day_name', 'month_name', 'season', 'is_weekend', 'is_rush_hour']].head()

,start_time,hour,day_name,month_name,season,is_weekend,is_rush_hour
0,2021-11-24 19:36:02,19,Wednesday,November,Fall,False,True
1,2017-08-08 08:52:45,8,Tuesday,August,Summer,False,True
2,2016-10-24 16:09:01,16,Monday,October,Fall,False,True
3,2022-12-23 14:05:02,14,Friday,December,Winter,False,False
4,2017-10-01 16:44:33,16,Sunday,October,Fall,True,True


## Step 9 — Map severity to human-readable labels

| Severity | Label |
|---|---|
| 1 | Low |
| 2 | Moderate |
| 3 | High |
| 4 | Critical |

Also add `is_high_severity` (severity ≥ 3) for hypothesis testing.

In [10]:
df = add_severity_label(df)
df['severity_label'].value_counts()

severity_label
Moderate    69901
High        16829
Critical     2348
Low           842
Name: count, dtype: int64

## Step 10 — Handle outliers

- Cap `distance_mi` at the 99th percentile (long-tail artefacts).
- Remove temperatures outside the physically possible range (-60 °F to 140 °F).

In [11]:
before = len(df)
df = handle_outliers(df)
print(f'Dropped {before - len(df):,} rows with impossible temperature values')
print('Distance distribution after capping:')
print(df['distance_mi'].describe().round(3))

Dropped 0 rows with impossible temperature values
Distance distribution after capping:
count    89920.000
mean         0.458
std          1.070
min          0.000
25%          0.000
50%          0.010
75%          0.383
max          6.659
Name: distance_mi, dtype: float64


## Step 11 — Drop exact duplicates and finalise

In [12]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Dropped {before - len(df):,} exact duplicates')
print(f'Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

Dropped 26 exact duplicates
Final shape: 89,894 rows × 48 columns


## Cleaning audit

Compare row and null counts before and after cleaning.

In [13]:
rows_final = len(df)
loss_pct = (rows_initial - rows_final) / rows_initial * 100
print(f'Initial rows : {rows_initial:,}')
print(f'Final rows   : {rows_final:,}')
print(f'Row loss     : {rows_initial - rows_final:,} ({loss_pct:.2f}%)')

remaining_nulls = df.isna().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print('\nRemaining nulls (should be empty or only in low-priority columns):')
print(remaining_nulls if not remaining_nulls.empty else '  none')

Initial rows : 100,000
Final rows   : 89,894
Row loss     : 10,106 (10.11%)



Remaining nulls (should be empty or only in low-priority columns):
  none


## Export cleaned dataset

In [14]:
# Intermediate save removed — single final save happens after post-cleaning fixes below.
print(f'Cleaning pipeline complete: {df.shape[0]:,} rows × {df.shape[1]} columns')
print('Proceeding to post-cleaning fixes before saving...')

Cleaning pipeline complete: 89,894 rows × 48 columns
Proceeding to post-cleaning fixes before saving...


---

## Post-cleaning fixes

Two known issues from QA review are addressed below. The original `df` is preserved — only **new columns are added**. The single output file `data/processed/cleaned_dataset.csv` is written at the end.

### Issue 1 — `distance_mi` zero values (~47% of rows)

`distance_mi` has a large block of `0.0` values. These are likely point-location accidents where road distance was not measured. We **flag** them rather than drop them.

In [15]:
# Issue 1 / Step 1: log how many rows have distance_mi == 0 before any change
zero_count = (df['distance_mi'] == 0).sum()
zero_pct = zero_count / len(df) * 100
print(f'distance_mi == 0 rows: {zero_count:,}')
print(f'percentage of dataset: {zero_pct:.2f}%')
print('top 10 distance_mi values:')
print(df['distance_mi'].value_counts().head(10))

distance_mi == 0 rows: 42,632
percentage of dataset: 47.42%
top 10 distance_mi values:
distance_mi
0.00000    42632
0.01000     3339
6.65905      899
0.01000      182
0.00900      162
0.00800      157
0.01100      152
0.02800      125
0.02600      124
0.02700      123
Name: count, dtype: int64


In [16]:
# Issue 1 / Step 2: add is_zero_distance flag column (do NOT drop rows)
df['is_zero_distance'] = df['distance_mi'] == 0
print('is_zero_distance value counts after adding:')
print(df['is_zero_distance'].value_counts())

is_zero_distance value counts after adding:
is_zero_distance
False    47262
True     42632
Name: count, dtype: int64


**Issue 1 resolution:** 42,632 rows (47%) had `distance_mi = 0`. These likely represent point-location accidents where road distance was not measured. Rows were retained and flagged with `is_zero_distance` for downstream analysis.

### Issue 2 — `duration_min` outliers (~9.5% of rows)

`duration_min` has a long right tail (max = 1440 min = 24 h). We compute the IQR threshold, flag long-duration accidents, and create a capped version for analysis — without modifying the original column.

In [17]:
# Issue 2 / Step 1: IQR diagnostics for duration_min before any change
q1 = df['duration_min'].quantile(0.25)
q3 = df['duration_min'].quantile(0.75)
iqr = q3 - q1
upper_iqr = q3 + 1.5 * iqr
outlier_count = (df['duration_min'] > upper_iqr).sum()

print(f'Q1 = {q1:.2f} min | Q3 = {q3:.2f} min | IQR = {iqr:.2f} min')
print(f'IQR upper threshold (Q3 + 1.5*IQR) = {upper_iqr:.2f} min')
print(f'rows above IQR upper threshold: {outlier_count:,}')

print('\ndescribe() for rows where duration_min > 600:')
print(df.loc[df['duration_min'] > 600, 'duration_min'].describe())

bins = [0, 60, 120, 360, 720, df['duration_min'].max() + 1]
labels = ['0-60', '60-120', '120-360', '360-720', '720+']
bucket_counts = pd.cut(df['duration_min'], bins=bins, labels=labels, right=False).value_counts().sort_index()
print('\nduration_min bucket distribution (minutes):')
print(bucket_counts)

Q1 = 30.00 min | Q3 = 122.30 min | IQR = 92.30 min
IQR upper threshold (Q3 + 1.5*IQR) = 260.75 min
rows above IQR upper threshold: 8,557

describe() for rows where duration_min > 600:
count    1067.000000
mean      821.328679
std       174.449448
min       600.650000
25%       704.641667
50%       793.033333
75%       883.025000
max      1440.000000
Name: duration_min, dtype: float64

duration_min bucket distribution (minutes):
duration_min
0-60       42747
60-120     23237
120-360    16697
360-720     6465
720+         748
Name: count, dtype: int64


In [18]:
# Issue 2 / Step 2: add is_long_duration flag (do NOT drop or cap original column)
df['is_long_duration'] = df['duration_min'] > 720
print('is_long_duration value counts after adding:')
print(df['is_long_duration'].value_counts())

is_long_duration value counts after adding:
is_long_duration
False    89149
True       745
Name: count, dtype: int64


In [19]:
# Issue 2 / Step 3: create duration_min_capped at 720; keep original duration_min untouched
df['duration_min_capped'] = df['duration_min'].clip(upper=720)

print('before (duration_min):')
print(f'  mean = {df["duration_min"].mean():.2f} | max = {df["duration_min"].max():.2f} | std = {df["duration_min"].std():.2f}')
print('after  (duration_min_capped):')
print(f'  mean = {df["duration_min_capped"].mean():.2f} | max = {df["duration_min_capped"].max():.2f} | std = {df["duration_min_capped"].std():.2f}')

before (duration_min):
  mean = 106.65 | max = 1440.00 | std = 125.00
after  (duration_min_capped):
  mean = 105.24 | max = 720.00 | std = 115.84


**Issue 2 resolution:** 8,557 rows (9.5%) had `duration_min` above the IQR outlier threshold. Accidents lasting more than 720 minutes (12 hours) were flagged with `is_long_duration`. A capped version `duration_min_capped` was created for statistical analysis to reduce skew, while the original column was preserved.

### Final save — `data/processed/cleaned_dataset.csv`

In [20]:
# Final step: report shape/columns, summarize new columns, verify quality, and save
new_columns = ['is_zero_distance', 'is_long_duration', 'duration_min_capped']

print(f'Final shape: {df.shape}')
print(f'Total columns: {len(df.columns)}')
print(f'Column list: {list(df.columns)}')

print('\n--- Change summary ---')
print(f'New columns added ({len(new_columns)}): {new_columns}')
print(f'Final null count: {df.isna().sum().sum()}')
print(f'Final duplicate count: {df.duplicated().sum()}')

PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f'\nSaved successfully to {PROCESSED_PATH.relative_to(PROJECT_ROOT)}')

Final shape: (89894, 51)
Total columns: 51
Column list: ['severity', 'start_time', 'end_time', 'start_lat', 'start_lng', 'distance_mi', 'street', 'city', 'county', 'state', 'zipcode', 'timezone', 'temperature_f', 'wind_chill_f', 'humidity', 'pressure_in', 'visibility_mi', 'wind_direction', 'wind_speed_mph', 'precipitation_in', 'weather_condition', 'amenity', 'bump', 'crossing', 'give_way', 'junction', 'no_exit', 'railway', 'roundabout', 'station', 'stop', 'traffic_calming', 'traffic_signal', 'sunrise_sunset', 'civil_twilight', 'duration_min', 'hour', 'day_of_week', 'day_name', 'month', 'month_name', 'year', 'date', 'season', 'is_weekend', 'is_rush_hour', 'severity_label', 'is_high_severity', 'is_zero_distance', 'is_long_duration', 'duration_min_capped']

--- Change summary ---
New columns added (3): ['is_zero_distance', 'is_long_duration', 'duration_min_capped']
Final null count: 0


Final duplicate count: 0



Saved successfully to data/processed/cleaned_dataset.csv
